In [3]:
from google import genai
import enum
import json
from time import sleep

import utils
from importlib import reload

In [ ]:
reload(utils)

In [2]:
with open('../ocr/gemini/gemini-key.txt', 'r') as f:
    api_key = f.read().strip()
client = genai.Client(api_key=api_key)

In [3]:
with open('prosody_prompt.txt', 'r') as f:
    PROMPT = f.read()

with open('prosody_align_prompt.txt', 'r') as f:
    PROMPT_ALIGN = f.read()


In [4]:
source_name = "exercises-in-latin-prosody"
source_year = 1823
difficulty = 'unknown'

In [5]:
with open('../data/semi_structured/prosody_questions.json', 'r') as f:
    textbook_dict = json.load(f)

with open('../data/semi_structured/prosody_key.json', 'r') as f:
    key_dict = json.load(f)   

In [7]:
part_name = "PART III"
chapter_name = "CHAPTER IX"
exercise_text = textbook_dict[part_name][chapter_name]
key_text = key_dict[part_name][chapter_name]

part_letter = part_name.split()[-1]
chap_letter = chapter_name.split()[-1]
question_id_prefix = f"{source_name}_{source_year}_part-{part_letter}_chap-{chap_letter}"

In [8]:
prompt = PROMPT_ALIGN.format(
    exercise_text,
    key_text
)

In [9]:
prompt

'I will give you a list of questions and their answer key. You should roughly separate the questions and answers into blocks and align the answers to their corresponding questions. If there are instructions on how to answer the question, or other information necessary to answer the question, put that in the question_instructions field.\n\nQuestions: The words in the following exercises, which are inclosed within brackets, are examples of the periphrasis, and are to be substituted for the corresponding word in the line. When two or more Italic words occur in a line, they must be omitted, and the meaning, which they are designed to convey, expressed by one word only. When there is only one word in a line printed in Italics, it is intended to be omitted, and its meaning expressed by a periphrasis.\n1.\nThus does the lioness rage when confined in a narrow\nden, And breaks her fierce teeth by biting her prison.\nSic leæna fremo (fera nobilis) in claustrum (enall.) parvus\nabditus,\nEt rabid

In [10]:
response = client.models.generate_content(
    model='gemini-2.0-flash',
    contents=prompt,
    config=utils.config_aligned_text_block
)

In [11]:
response.text

'[\n  {\n    "question_text": "Thus does the lioness rage when confined in a narrow\\nden, And breaks her fierce teeth by biting her prison.\\nSic leæna fremo (fera nobilis) in claustrum (enall.) parvus\\nabditus,\\nEt rabidus dens frango carcere præmorso.",\n    "answer_text": "Sic fremit in parvis fera nobilis abdita claustris,\\nEt frangit rabidos præmorso carcere dentes.",\n    "question_instructions": "The words in the following exercises, which are inclosed within brackets, are examples of the periphrasis, and are to be substituted for the corresponding word in the line. When two or more Italic words occur in a line, they must be omitted, and the meaning, which they are designed to convey, expressed by one word only. When there is only one word in a line printed in Italics, it is intended to be omitted, and its meaning expressed by a periphrasis."\n  },\n  {\n    "question_text": "Whither shall I be carried? where shall I seek comfort\\nin my affliction? No anchor now holds my ba

In [12]:
resp_text = response.text
resp_text = resp_text.replace('null', 'None')
resp_list = eval(resp_text)

In [13]:
resp_list

[{'question_text': 'Thus does the lioness rage when confined in a narrow\nden, And breaks her fierce teeth by biting her prison.\nSic leæna fremo (fera nobilis) in claustrum (enall.) parvus\nabditus,\nEt rabidus dens frango carcere præmorso.',
  'answer_text': 'Sic fremit in parvis fera nobilis abdita claustris,\nEt frangit rabidos præmorso carcere dentes.',
  'question_instructions': 'The words in the following exercises, which are inclosed within brackets, are examples of the periphrasis, and are to be substituted for the corresponding word in the line. When two or more Italic words occur in a line, they must be omitted, and the meaning, which they are designed to convey, expressed by one word only. When there is only one word in a line printed in Italics, it is intended to be omitted, and its meaning expressed by a periphrasis.'},
 {'question_text': 'Whither shall I be carried? where shall I seek comfort\nin my affliction? No anchor now holds my bark.\nQuò feror? unde (lapsis rebus)

roughly align text blocks (groups of aligned questions)

In [1]:
# load text blocks
with open('../data/structured/prosody_aligned_text_blocks.json', 'r') as f:
    all_blocks = json.load(f)

FileNotFoundError: [Errno 2] No such file or directory: '../data/structured/prosody_aligned_text_blocks.json'

In [4]:
with open('../data/structured/prosody_aligned_text_blocks_backup.json', 'r') as f:
    all_blocks = json.load(f)

In [8]:
import pandas as pd
flat_blocks = []
for part_name, chapter_dict in all_blocks.items():
    for chapter_name, block_list in chapter_dict.items():
        for block in block_list:
            #print(block)
            if type(block) == str:
                continue
            block['part_name'] = part_name
            block['chapter_name'] = chapter_name
            flat_blocks.append(block)

df = pd.DataFrame(flat_blocks)
df.to_csv('prosody_aligned_text_blocks.tsv', sep='\t', index=False)



In [ ]:
# part -> chapter -> block;
# because questions in one chapter are related and 
# may share instructions

for part_name, chapter_dict in textbook_dict.items():
    print(part_name)
    if part_name not in all_blocks:
        all_blocks[part_name] = {}
    for chapter_name, chapter_text in chapter_dict.items():
        #if part_name == "PART I" and chapter_name == "CHAPTER I":
        #    continue 
        print('  '+chapter_name)

        # initialize empty list for this chapter
        # if it's not already in there
        if chapter_name not in all_blocks[part_name]:
            all_blocks[part_name][chapter_name] = []
        else:
            print(f"Chapter {chapter_name} already in {part_name}")
            print("Skipping...")
            continue

        part_letter = part_name.split()[-1]
        chap_letter = chapter_name.split()[-1]
        question_id_prefix = f"{source_name}_{source_year}_part-{part_letter}_chap-{chap_letter}"

        exercise_text = textbook_dict[part_name][chapter_name]
        
        if part_name in key_dict and chapter_name in key_dict[part_name]:
            key_text = key_dict[part_name][chapter_name]
        else:
            print(f"No key text for {part_name} {chapter_name}")
            print("Skipping...")
            continue

        prompt = PROMPT_ALIGN.format(
            exercise_text,
            key_text
        )

        response = client.models.generate_content(
            #model='gemini-2.0-flash',
            model='gemini-2.5-pro',
            thinkingBudget = -1,
            contents=prompt,
            config=utils.config_aligned_text_block
        )

        resp_text = response.text
        resp_text = resp_text.replace('null', 'None')
        
        try:
            resp_list = eval(resp_text)
        except Exception as e:
            print(f"Error evaluating response: {e}")
            print(f"Response: {resp_text}")
            # just save as str 
            resp_list = [resp_text]

        all_blocks[part_name][chapter_name] = resp_list 

        # dump intermediate to file
        with open('../data/structured/prosody_aligned_text_blocks.json', 'w') as f:
            json.dump(all_blocks, f, indent=4)

        sleep(0.2)


PART I
  CHAPTER I
Chapter CHAPTER I already in PART I
Skipping...
  CHAPTER II
Chapter CHAPTER II already in PART I
Skipping...
  CHAPTER III
Chapter CHAPTER III already in PART I
Skipping...
  CHAPTER IV
Chapter CHAPTER IV already in PART I
Skipping...
PART II
  CHAPTER I
Chapter CHAPTER I already in PART II
Skipping...
  CHAPTER II
Chapter CHAPTER II already in PART II
Skipping...
  CHAPTER III
Chapter CHAPTER III already in PART II
Skipping...
PART III
  CHAPTER I
Chapter CHAPTER I already in PART III
Skipping...
  CHAPTER II
Error evaluating response: unterminated string literal (detected at line 273) (<string>, line 273)
Response: [
  {
    "question_text": "No. 8.\nHaud sic magni conditor orbis;\nHuic ex alto cuncta tuenti\nNullâ terræ mole resistunt,\nNon nox atris nubibus obstat.",
    "answer_text": "Haud sic | magni | conditor | orbis;\nHuic ex | alto | cuncta tuenti\nNulla | terrae | mole resistunt,\nNon nox | atris | nubibus | obstat.",
    "question_instructions": "The fi

classify questions, format into json

In [ ]:
all_questions = []
all_questions.extend(resp_list)

for part_name, chapter_dict in textbook_dict.items():
    print(part_name)
    for chapter_name, chapter_text in chapter_dict.items():
        if part_name == "PART I" and chapter_name == "CHAPTER I":
            continue 
        print('  '+chapter_name)

        part_letter = part_name.split()[-1]
        chap_letter = chapter_name.split()[-1]
        question_id_prefix = f"{source_name}_{source_year}_part-{part_letter}_chap-{chap_letter}"

        exercise_text = textbook_dict[part_name][chapter_name]
        key_text = key_dict[part_name][chapter_name]

        prompt = PROMPT.format(
            source_name,
            source_year,
            difficulty,
            question_id_prefix,
            exercise_text,
            key_text
        )

        response = client.models.generate_content(
            model='gemini-2.0-flash',
            contents=prompt,
            config=utils.config
        )

        resp_text = response.text
        resp_text = resp_text.replace('null', 'None')
        resp_list = eval(resp_text)

        all_questions.extend(resp_list)

        # dump intermediate to file
        with open('../data/structured/prosody_questions.json', 'w') as f:
            json.dump(all_questions, f, indent=4)

        sleep(0.2)

PART I
  CHAPTER II


FileNotFoundError: [Errno 2] No such file or directory: '../data/stuctured/prosody_questions.json'

In [29]:
with open('../data/structured/prosody_questions.json', 'w') as f:
    json.dump(all_questions, f, indent=4)